# BLIP Training

In [1]:
!pip install -q \
  transformers \
  peft \
  "datasets<3.0.0" \
  evaluate \
  rouge_score \
  bert_score \
  nltk \
  huggingface_hub \
  torch \
  torchvision \
  tqdm \
  pillow \
  pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from datasets import load_dataset
from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_scheduler,
)

import evaluate
import nltk

try:
    nltk.download("wordnet")
    nltk.download("omw-1.4")
except Exception:
    pass

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "Salesforce/blip-image-captioning-base"

print(f"Using device: {device}")
print(f"Using dtype: {dtype}")

Using device: cuda
Using dtype: torch.float16


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
processor = BlipProcessor.from_pretrained(model_id)

full_dataset = load_dataset(
    "nlphuji/flickr30k",
    split="test",
    trust_remote_code=True
)

train_data = full_dataset.filter(lambda x: x["split"] == "train")
val_data = full_dataset.filter(lambda x: x["split"] == "val")
test_data = full_dataset.filter(lambda x: x["split"] == "test")

print(f"Train size: {len(train_data)}")
print(f"Val size: {len(val_data)}")
print(f"Test size: {len(test_data)}")

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
class Flickr30kCaptionDataset(Dataset):
    def __init__(self, hf_dataset, processor, max_length=50):
        self.dataset = hf_dataset
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def _get_captions(self, captions):
        if isinstance(captions, str):
            try:
                captions = json.loads(captions)
            except Exception:
                captions = [captions]
        return captions

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"].convert("RGB")

        captions = self._get_captions(item["caption"])
        caption = random.choice(captions)

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        labels = encoding["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        encoding["labels"] = labels

        return encoding

In [ ]:
def make_loader(hf_data, processor, batch_size=32, shuffle=True, max_length=50):
    ds = Flickr30kCaptionDataset(
        hf_dataset=hf_data,
        processor=processor,
        max_length=max_length
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

train_loader_32 = make_loader(train_data, processor, batch_size=32, shuffle=True)
train_loader_large = make_loader(train_data, processor, batch_size=128, shuffle=True)

print("Dataloaders ready.")

In [ ]:
def generate_caption(model, image, processor, device, max_new_tokens=50, num_beams=1):
    model.eval()

    image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams
        )

    caption = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    return caption

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

def normalize_captions(captions):
    if isinstance(captions, str):
        try:
            captions = json.loads(captions)
        except Exception:
            captions = [captions]
    return captions

def evaluate_captioning(
    model,
    hf_dataset,
    processor,
    device,
    batch_size=16,
    max_new_tokens=50,
    num_beams=1,
    max_eval_samples=None
):
    model.eval()
    model.to(device)

    predictions = []
    references = []

    n = len(hf_dataset) if max_eval_samples is None else min(max_eval_samples, len(hf_dataset))

    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc="Evaluating BLIP captions"):
            end = min(start + batch_size, n)
            batch = hf_dataset[start:end]

            images = [img.convert("RGB") for img in batch["image"]]

            inputs = processor(
                images=images,
                return_tensors="pt",
                padding=True
            ).to(device)

            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams
            )

            batch_predictions = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )

            batch_predictions = [p.strip() for p in batch_predictions]

            batch_refs = [
                normalize_captions(caps)
                for caps in batch["caption"]
            ]

            predictions.extend(batch_predictions)
            references.extend(batch_refs)

    # BLEU expects list[str] predictions and list[list[str]] references
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=references,
        max_order=4
    )

    # ROUGE expects one reference string per prediction.
    # Use the first reference for ROUGE-L to keep it simple and consistent.
    rouge_refs = [refs[0] for refs in references]
    rouge = rouge_metric.compute(
        predictions=predictions,
        references=rouge_refs
    )

    meteor = meteor_metric.compute(
        predictions=predictions,
        references=rouge_refs
    )

    bertscore = bertscore_metric.compute(
        predictions=predictions,
        references=rouge_refs,
        lang="en"
    )

    return {
        "BLEU-4": round(bleu["bleu"], 4),
        "ROUGE-L": round(rouge["rougeL"], 4),
        "METEOR": round(meteor["meteor"], 4),
        "BERTScore": round(float(np.mean(bertscore["f1"])), 4)
    }, predictions, references

In [ ]:
def train_blip_model(
    model,
    train_loader,
    epochs=1,
    lr=5e-5,
    weight_decay=0.01,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0
):
    model.to(device)

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

    num_update_steps_per_epoch = len(train_loader) // gradient_accumulation_steps
    num_training_steps = epochs * num_update_steps_per_epoch

    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )

    history = {
        "step_loss": [],
        "epoch_loss": []
    }

    model.train()

    for epoch in range(epochs):
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
        epoch_loss = 0.0

        optimizer.zero_grad()

        for step, batch in enumerate(pbar):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss / gradient_accumulation_steps

            loss.backward()

            if (step + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    max_grad_norm
                )

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            true_loss = loss.item() * gradient_accumulation_steps
            history["step_loss"].append(true_loss)
            epoch_loss += true_loss

            pbar.set_postfix({"loss": true_loss})

        history["epoch_loss"].append(epoch_loss / len(train_loader))

    return model, history

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title="BLIP Training Loss"):
    plt.figure(figsize=(10, 5))

    plt.plot(history["step_loss"], label="Step Loss", alpha=0.4)

    if len(history["epoch_loss"]) > 0:
        steps_per_epoch = max(1, len(history["step_loss"]) // len(history["epoch_loss"]))
        epoch_x = [
            i * steps_per_epoch + steps_per_epoch - 1
            for i in range(len(history["epoch_loss"]))
        ]
        plt.plot(
            epoch_x,
            history["epoch_loss"],
            label="Epoch Loss Avg",
            marker="o",
            linewidth=2
        )

    plt.title(title)
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

In [ ]:
results_table = {}

def log_result(name, metrics):
    results_table[name] = metrics
    print(f"\n--- {name} ---")
    for k, v in metrics.items():
        print(f"{k}: {v}")

In [ ]:
def setup_blip_linear_probe():
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    for param in model.parameters():
        param.requires_grad = False

    # Train the language modeling head if available.
    # This adapts the output vocabulary distribution while keeping the main model frozen.
    trainable_keywords = [
        "cls",
        "lm_head"
    ]

    for name, param in model.named_parameters():
        if any(key in name for key in trainable_keywords):
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

    return model

# Linear Probe

In [ ]:
probe_model = setup_blip_linear_probe()

probe_model, probe_history = train_blip_model(
    probe_model,
    train_loader_large,
    epochs=10,
    lr=1e-4,
    gradient_accumulation_steps=1
)

plot_history(probe_history, title="BLIP Frozen Probe Training Loss")

probe_metrics, probe_preds, probe_refs = evaluate_captioning(
    probe_model,
    test_data,
    processor,
    device,
    batch_size=16,
    max_new_tokens=50,
    num_beams=1,
    max_eval_samples=None
)

log_result("Linear Probe", probe_metrics)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

def upload_blip_to_hf(model, processor, repo_name):
    try:
        model.push_to_hub(repo_name)
        processor.push_to_hub(repo_name)
        print(f"Uploaded model and processor to Hugging Face Hub: {repo_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

In [ ]:
upload_blip_to_hf(probe_model, processor, "bdanko/blip-flickr30k-probe")

# Partial Fine Tune

In [ ]:
def setup_blip_partial_finetune():
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze final vision encoder block.
    try:
        for param in model.vision_model.encoder.layers[-1].parameters():
            param.requires_grad = True
        print("Unfroze final vision encoder block.")
    except Exception as e:
        print(f"Could not unfreeze final vision block: {e}")

    # Unfreeze final text decoder block.
    try:
        for param in model.text_decoder.bert.encoder.layer[-1].parameters():
            param.requires_grad = True
        print("Unfroze final text decoder block.")
    except Exception as e:
        print(f"Could not unfreeze final decoder block: {e}")

    # Unfreeze LM head / classifier layers.
    for name, param in model.named_parameters():
        if "cls" in name or "lm_head" in name:
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

    return model

In [ ]:
partial_model = setup_blip_partial_finetune()

partial_model, partial_history = train_blip_model(
    partial_model,
    train_loader_32,
    epochs=1,
    lr=5e-5,
    gradient_accumulation_steps=1
)

plot_history(partial_history, title="BLIP Partial Fine-Tune Training Loss")

partial_metrics, partial_preds, partial_refs = evaluate_captioning(
    partial_model,
    test_data,
    processor,
    device,
    batch_size=16,
    max_new_tokens=50,
    num_beams=1,
    max_eval_samples=None
)

log_result("Partial Fine-tune", partial_metrics)

# LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

def print_linear_module_names(model, limit=80):
    names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            names.append(name)

    print(f"Found {len(names)} Linear modules.")
    for name in names[:limit]:
        print(name)

# Optional inspection:
# temp_model = BlipForConditionalGeneration.from_pretrained(model_id)
# print_linear_module_names(temp_model)

In [ ]:
def setup_blip_lora():
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    config = LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=[
            "query",
            "key",
            "value"
        ],
        lora_dropout=0.1,
        bias="none"
    )

    model = get_peft_model(model, config)
    model.print_trainable_parameters()

    return model

In [ ]:
lora_model = setup_blip_lora()

lora_model, lora_history = train_blip_model(
    lora_model,
    train_loader_32,
    epochs=1,
    lr=5e-5,
    gradient_accumulation_steps=1
)

plot_history(lora_history, title="BLIP LoRA Training Loss")

lora_metrics, lora_preds, lora_refs = evaluate_captioning(
    lora_model,
    test_data,
    processor,
    device,
    batch_size=16,
    max_new_tokens=50,
    num_beams=1,
    max_eval_samples=None
)

log_result("LoRA", lora_metrics)

# Full Fine Tune

In [ ]:
full_model = BlipForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype
)

for param in full_model.parameters():
    param.requires_grad = True

full_model, full_history = train_blip_model(
    full_model,
    train_loader_32,
    epochs=1,
    lr=5e-6,
    gradient_accumulation_steps=1
)

plot_history(full_history, title="BLIP Full Fine-Tune Training Loss")

full_metrics, full_preds, full_refs = evaluate_captioning(
    full_model,
    test_data,
    processor,
    device,
    batch_size=16,
    max_new_tokens=50,
    num_beams=1,
    max_eval_samples=None
)

log_result("Full Fine-tune", full_metrics)

# Results

In [ ]:
df = pd.DataFrame(results_table).T
df = df[["BLEU-4", "ROUGE-L", "METEOR", "BERTScore"]]

print("BLIP Fine-Tuning Results")
display(df)

df.to_csv("blip_flickr30k_results.csv")